# MIT License

Copyright (c) 2024 Social Science Research Notebooks

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.

In [ ]:
# Setup for Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !pip install -q pymc arviz pandas numpy matplotlib seaborn
    print('Packages installed.')


# Bayesian Analysis with PyMC In-Depth

Bayesian statistics offers a powerful alternative to frequentist (classic) statistics. Instead of calculating a single 'p-value', Bayesian analysis allows us to estimate the **probability distribution** of our parameters given the data.

In this notebook, we will use **PyMC** (formerly PyMC3), the standard library for Probabilistic Programming in Python, to perform a simple Bayesian Linear Regression.

**Key Concepts:**
1.  **Prior**: What we believe about the parameters *before* seeing data.
2.  **Likelihood**: How likely the data is given the parameters.
3.  **Posterior**: What we believe about the parameters *after* seeing data (Prior + Likelihood).


In [ ]:
import pymc as pm
import arviz as az
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)
print(f"PyMC version: {pm.__version__}")

## 1. Synthetic Data Generation

Let's simulate a study on **Income vs. Happiness**.
- **Income (X)**: Normalized score (0-10).
- **Happiness (Y)**: Self-reported score (0-100).
- True Relationship: $Happiness = 20 + 5 \times Income + Noise$

In [ ]:
n_samples = 100
true_alpha = 20
true_beta = 5
sigma = 10

income = np.random.uniform(0, 10, n_samples)
happiness = true_alpha + true_beta * income + np.random.normal(0, sigma, n_samples)

data = pd.DataFrame({'Income': income, 'Happiness': happiness})

plt.figure(figsize=(8, 5))
sns.scatterplot(x='Income', y='Happiness', data=data)
plt.title('Simulated Data: Income vs Happiness')
plt.show()

## 2. Defining the Bayesian Model

We need to define priors for our parameters:
- $\alpha$ (Intercept): We expect it to be positive, maybe around 30? Let's use a Normal distribution.
- $\beta$ (Slope): We expect a positive relationship. Normal distribution.
- $\sigma$ (Error): Must be positive. Half-Normal distribution.

$$ Y \sim N(\mu, \sigma) $$
$$ \mu = \alpha + \beta X $$

In [ ]:
with pm.Model() as model:
    # 1. Define Priors
    # We are somewhat uncertain, so we give wide priors (large sigma)
    alpha = pm.Normal('alpha', mu=0, sigma=50)
    beta = pm.Normal('beta', mu=0, sigma=20)
    sigma_obs = pm.HalfNormal('sigma', sigma=20)

    # 2. Define Linear Relationship
    mu = alpha + beta * data['Income'].values

    # 3. Define Likelihood (The expected distribution of data)
    Y_obs = pm.Normal('Y_obs', mu=mu, sigma=sigma_obs, observed=data['Happiness'].values)

    print("Model structure defined.")

## 3. Sampling (MCMC)

Now we use the **NUTS (No-U-Turn Sampler)** algorithm to draw samples from the posterior distribution. This essentially 'explores' the parameter space to find the most likely values.

In [ ]:
with model:
    # Draw 1000 posterior samples using NUTS sampling
    # tune=1000 means it spends 1000 steps adjusting step size before recording
    idata = pm.sample(draws=1000, tune=1000, chains=2, return_inferencedata=True, progressbar=False)

print("Sampling complete.")

## 4. Analyzing Results

We can inspect the trace plots. Ideally, we want:
1.  **Stationarity**: The trace (right) should look like a fuzzy caterpillar, not wandering up or down.
2.  **Convergence**: The distributions (left) for all chains should overlap.

In [ ]:
az.plot_trace(idata)
plt.tight_layout()
plt.show()

### Posterior Summary

The summary table gives us:
- **mean**: The expected value of the parameter.
- **hdi_3% / hdi_97%**: The **Highest Density Interval** (Credible Interval). There is a 94% probability the true parameter lies within this range (unlike the frequentist CI, which is about repeated sampling).

In [ ]:
summary = az.summary(idata, hdi_prob=0.95)
print(summary)

print(f"\nTrue Alpha: {true_alpha}")
print(f"True Beta: {true_beta}")

## 5. Visualizing the Posterior Predictive

Instead of a single regression line, Bayesian analysis gives us a *distribution* of possible lines. Let's plot a few hundred of them to visualize our uncertainty.

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(x='Income', y='Happiness', data=data, label='Data')

# Extract samples
posterior_alpha = idata.posterior['alpha'].values.flatten()
posterior_beta = idata.posterior['beta'].values.flatten()

# Plot first 100 credible regression lines
x_range = np.linspace(0, 10, 100)
for i in range(100):
    y_est = posterior_alpha[i] + posterior_beta[i] * x_range
    plt.plot(x_range, y_est, color='k', alpha=0.1)

plt.title('Uncertainty in Regression Lines (Posterior Samples)')
plt.xlabel('Income')
plt.ylabel('Happiness')
plt.legend()
plt.show()